# Trabajo Práctico: Matrices Insumo-Producto (Modelo de Leontief)
**Maestría en Ciencia de Datos - Universidad de San Andrés**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

---
## Consigna 1: Función `calcularLeontief(A)`

La matriz de Leontief se define como:
$$L = (I - A)^{-1}$$
Donde $A$ es la matriz de coeficientes técnicos e $I$ es la identidad del mismo tamaño.

In [ ]:
def calcularLeontief(A):
    """
    Calcula la matriz de Leontief L = (I - A)^{-1}.
    
    Parámetros:
        A (ndarray): Matriz de coeficientes técnicos de tamaño n x n.
    
    Retorna:
        L (ndarray): Matriz de Leontief de tamaño n x n.
    """
    n = A.shape[0]
    I = np.eye(n)
    return np.linalg.inv(I - A)

---
## Consigna 2: Resolver el sistema con una A particular

Dada la matriz $A$ con un coeficiente técnico de 1.0 en la diagonal del sector 2:

In [ ]:
A2 = np.array([
    [0.30, 0.00, 0.10],
    [0.05, 1.00, 0.20],
    [0.10, 0.15, 0.10]
])

d2 = np.array([100, 100, 300])

L2 = calcularLeontief(A2)
p2 = L2 @ d2

print("Matriz de Leontief L:")
print(L2)
print("\nVector de producción p:")
print(p2)

### Interpretación económica

Los valores negativos en $L$ y en $p$ **no tienen sentido económico**: no es posible producir cantidades negativas de un bien.

El problema radica en el coeficiente técnico $a_{22} = 1.0$: el sector 2 consume exactamente toda su propia producción para producirse a sí mismo, sin margen para demanda externa. Esto hace que $(I - A)$ sea **singular o casi singular**, llevando a valores extremos y negativos al invertirla.

Para que el modelo de Leontief tenga solución económicamente válida, todos los coeficientes técnicos de cada columna deben sumar **estrictamente menos que 1** (condición de viabilidad económica).

---
## Consigna 3: Función `deltaProduccion(L, delta_d)`

A partir del modelo $p = Ld$, si la demanda externa cambia en $\Delta d$, la variación de producción es:
$$\Delta p = L \cdot \Delta d$$

In [ ]:
def deltaProduccion(L, delta_d):
    """
    Calcula la variación de producción ante un shock en la demanda externa.
    
    Parámetros:
        L (ndarray): Matriz de Leontief.
        delta_d (ndarray): Vector de variación de demanda externa.
    
    Retorna:
        delta_p (ndarray): Vector de variación de producción.
    """
    return L @ delta_d

In [ ]:
# Usamos una A válida del ejemplo introductorio del enunciado
A3 = np.array([
    [0.10, 0.15, 0.12],
    [0.20, 0.00, 0.30],
    [0.25, 0.40, 0.20]
])

L3 = calcularLeontief(A3)

# Shock unitario en la demanda del producto 3
delta_d3 = np.array([0, 0, 1])
delta_p3 = deltaProduccion(L3, delta_d3)

print("Variación de producción ante un shock de 1 unidad en d3:")
print(f"  Δp1 = {delta_p3[0]:.4f}")
print(f"  Δp2 = {delta_p3[1]:.4f}")
print(f"  Δp3 = {delta_p3[2]:.4f}")

---
## Consigna 3 bis: Secuencia de shocks sobre el producto 3

In [ ]:
shocks = np.arange(-10, 21, 1)  # shocks de -10 a 20
delta_ps = []

for shock in shocks:
    delta_d = np.array([0, 0, shock])
    delta_p = deltaProduccion(L3, delta_d)
    delta_ps.append(delta_p)

delta_ps = np.array(delta_ps)

plt.figure(figsize=(10, 5))
plt.plot(shocks, delta_ps[:, 0], label='Δp1', marker='o', markersize=3)
plt.plot(shocks, delta_ps[:, 1], label='Δp2', marker='s', markersize=3)
plt.plot(shocks, delta_ps[:, 2], label='Δp3', marker='^', markersize=3)
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.xlabel('Shock en demanda externa del Producto 3')
plt.ylabel('Variación de producción (Δp)')
plt.title('Respuesta de cada sector ante shocks en la demanda del Producto 3')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretación

- La relación entre el shock y $\Delta p$ es **estrictamente lineal** para todos los sectores. Esto se justifica directamente desde el modelo: $\Delta p = L \cdot \Delta d$, donde $L$ es constante. Al variar solo la magnitud del shock en $d_3$, la respuesta de cada sector escala linealmente con ese factor.
- El sector con mayor pendiente (mayor sensibilidad) ante cambios en $d_3$ es el que tiene el coeficiente $L_{i3}$ más alto en la matriz de Leontief.

---
## Consigna 4: Función `calcularCoeficientes(Z, totales)`

Los coeficientes técnicos se calculan como $A = Z \cdot P^{-1}$, donde $P$ es la matriz diagonal de totales producidos.

In [ ]:
def calcularCoeficientes(Z, totales):
    """
    Calcula la matriz de coeficientes técnicos A = Z * P^{-1}.
    
    Parámetros:
        Z (ndarray): Matriz de flujos monetarios entre sectores.
        totales (ndarray): Vector con el total producido por cada sector.
    
    Retorna:
        A (ndarray): Matriz de coeficientes técnicos.
    """
    P_inv = np.diag(1.0 / totales)
    return Z @ P_inv

---
## Consigna 5: Coeficientes técnicos y Leontief para una economía concreta

In [ ]:
Z5 = np.array([
    [350,   0,   0],
    [ 50, 250, 150],
    [200, 150, 550]
], dtype=float)

totales5 = np.array([1000, 500, 1000], dtype=float)

A5 = calcularCoeficientes(Z5, totales5)
L5 = calcularLeontief(A5)

print("Matriz de coeficientes técnicos A:")
print(A5)
print("\nMatriz de Leontief L:")
print(L5)

---
## Consigna 5 bis: Visualización con seaborn (heatmaps)

In [ ]:
sector_labels = ['S1', 'S2', 'S3']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(A5, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=sector_labels, yticklabels=sector_labels,
            ax=axes[0], cbar_kws={'label': 'Coeficiente'})
axes[0].set_title('Matriz de Coeficientes Técnicos (A)\nEfectos directos')
axes[0].set_xlabel('Sector que produce')
axes[0].set_ylabel('Sector que consume')

sns.heatmap(L5, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=sector_labels, yticklabels=sector_labels,
            ax=axes[1], cbar_kws={'label': 'Multiplicador'})
axes[1].set_title('Matriz de Leontief (L)\nEfectos totales (directos + indirectos)')
axes[1].set_xlabel('Sector que recibe demanda')
axes[1].set_ylabel('Sector que produce')

plt.tight_layout()
plt.show()

### Interpretación

- **Matriz A**: muestra las dependencias directas e inmediatas. Por ejemplo, $a_{31} = 0.20$ indica que para producir 1 unidad en S1 se requieren 0.20 unidades de S3 directamente.
- **Matriz L**: los valores son sistemáticamente mayores que en A, especialmente en la diagonal (siempre ≥ 1). Esto refleja que al considerar todos los efectos en cadena (sector A necesita a B, B necesita a C, C necesita a A...), el requerimiento total es mayor que el directo.
- Las celdas que más se intensifican entre A y L revelan los sectores con mayor **efecto multiplicador indirecto** en la economía.

---
## Consigna 6: Función `calcularLeontiefInterregional`

La fórmula del modelo de dos regiones (fijando $\Delta d^s = 0$) es:
$$\Delta p^r = (I - A^{rr} - A^{rs}(I - A^{ss})^{-1}A^{sr})^{-1} \Delta d^r$$

In [ ]:
def calcularLeontiefInterregional(Arr, Ars, Ass, Asr):
    """
    Calcula la matriz de Leontief interregional para la región r,
    considerando el efecto de ida y vuelta con la región s.
    
    Implementa la ecuación 6:
        L_inter = (I - Arr - Ars * (I - Ass)^{-1} * Asr)^{-1}
    
    Parámetros:
        Arr (ndarray): Coeficientes técnicos intra-regionales de r (n x n).
        Ars (ndarray): Coeficientes técnicos de s hacia r (n x m).
        Ass (ndarray): Coeficientes técnicos intra-regionales de s (m x m).
        Asr (ndarray): Coeficientes técnicos de r hacia s (m x n).
    
    Retorna:
        L_inter (ndarray): Matriz de Leontief interregional (n x n).
    """
    n_r = Arr.shape[0]
    n_s = Ass.shape[0]
    Ir = np.eye(n_r)
    Is = np.eye(n_s)
    return np.linalg.inv(Ir - Arr - Ars @ np.linalg.inv(Is - Ass) @ Asr)

---
## Consigna 7: Caso real — Argentina y Brasil (CEPAL 2011)

In [ ]:
# ── Lectura del archivo ──────────────────────────────────────────────────────
data = pd.read_excel('matrizlatina2011_compressed_0.xlsx', sheet_name='LAC_IOT_2011')

arg_cols = [c for c in data.columns if str(c).startswith('ARG')]
bra_cols = [c for c in data.columns if str(c).startswith('BRA')]

# Filas: ARG = 0-39, BRA = 40-79
Zrr = data.loc[0:39,  arg_cols].values   # ARG -> ARG
Zrs = data.loc[0:39,  bra_cols].values   # ARG -> BRA
Zsr = data.loc[40:79, arg_cols].values   # BRA -> ARG
Zss = data.loc[40:79, bra_cols].values   # BRA -> BRA

totales_r = data.loc[0:39,  'Output'].values   # Total producido ARG
totales_s = data.loc[40:79, 'Output'].values   # Total producido BRA

nombre_sector = data['Sector'][0:40].values

print("Datos cargados correctamente.")
print(f"Sectores ARG: {len(totales_r)}, Sectores BRA: {len(totales_s)}")

In [ ]:
# ── Coeficientes técnicos ────────────────────────────────────────────────────
# Para matrices interregionales, se divide por los totales del país que PRODUCE
# (el que está en las columnas)
Arr = calcularCoeficientes(Zrr, totales_r)   # divide por totales ARG
Ass = calcularCoeficientes(Zss, totales_s)   # divide por totales BRA
Ars = calcularCoeficientes(Zrs, totales_s)   # ARG->BRA: divide por totales BRA
Asr = calcularCoeficientes(Zsr, totales_r)   # BRA->ARG: divide por totales ARG

print("Coeficientes técnicos calculados.")
print(f"Arr: suma máx de columna = {Arr.sum(axis=0).max():.4f}")
print(f"Ass: suma máx de columna = {Ass.sum(axis=0).max():.4f}")

In [ ]:
# ── Visualización: Top 10 sectores con mayor interrelación ARG-BRA ───────────
inter_A_12 = Ars.sum(axis=1)   # flujo de cada sector ARG hacia BRA
inter_A_21 = Asr.sum(axis=1)   # flujo de cada sector BRA hacia ARG

df_inter_top10 = pd.DataFrame({
    'Sector':  nombre_sector,
    'Arg → Bra': inter_A_12,
    'Bra → Arg': inter_A_21,
    'Total':   inter_A_12 + inter_A_21
}).nlargest(10, 'Total')

df_melted = df_inter_top10.melt(
    id_vars='Sector',
    value_vars=['Arg → Bra', 'Bra → Arg'],
    var_name='Flujo',
    value_name='Interrelación'
)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_melted, y='Sector', x='Interrelación', hue='Flujo', orient='h')
plt.title('Top 10 sectores con mayor interrelación entre Argentina y Brasil')
plt.xlabel('Suma de coeficientes técnicos interregionales')
plt.ylabel('')
plt.legend(title='Dirección del flujo')
plt.tight_layout()
plt.show()

In [ ]:
# ── Matrices de Leontief ─────────────────────────────────────────────────────
L_simple        = calcularLeontief(Arr)
L_interregional = calcularLeontiefInterregional(Arr, Ars, Ass, Asr)

print("Matrices de Leontief calculadas.")

In [ ]:
# ── Shocks en P1 (Argentina) ─────────────────────────────────────────────────
# s05: shock negativo del 10%
# s06, s07, s08: shock positivo del 3.3% cada uno
delta_d_r = np.zeros(40)
delta_d_r[4] = -0.10  * totales_r[4]   # s05
delta_d_r[5] =  0.033 * totales_r[5]   # s06
delta_d_r[6] =  0.033 * totales_r[6]   # s07
delta_d_r[7] =  0.033 * totales_r[7]   # s08

# Variación de producción en P1
delta_p_simple = deltaProduccion(L_simple, delta_d_r)
delta_p_inter  = deltaProduccion(L_interregional, delta_d_r)

# Variación de producción en P2 (Brasil), propagada desde el shock en ARG
Is = np.eye(40)
delta_p_s_inter = np.linalg.inv(Is - Ass) @ Asr @ delta_p_inter

print("Variación total P1 - modelo simple:        ", delta_p_simple.sum())
print("Variación total P1 - modelo interregional: ", delta_p_inter.sum())
print("Variación total P2 - modelo interregional: ", delta_p_s_inter.sum())

### Análisis del shock

- El **modelo simple** considera únicamente los efectos internos de Argentina, ignorando los lazos con Brasil.
- El **modelo interregional** incorpora el efecto de ida y vuelta: el shock en ARG afecta a BRA (vía $A^{sr}$), y BRA responde amplificando el efecto de vuelta hacia ARG (vía $A^{rs}$).
- La **variación en P2 (Brasil)** es un efecto secundario del shock en Argentina: aunque el shock se origina en ARG, Brasil también se ve afectado a través de sus vínculos comerciales intersectoriales.

---
## Consigna 7 bis: Análisis gráfico con secuencia de intensidades

In [ ]:
# Intensidades del shock sobre s05: -2%, -4%, -6%, -8%, -10%
# Los shocks sobre s06, s07, s08 se mantienen proporcionales:
# si s05 va al -10% -> s06/s07/s08 van al +3.3%, entonces la proporción es 3.3/10 = 0.33

intensidades_s05 = np.array([-0.02, -0.04, -0.06, -0.08, -0.10])

total_p1_simple = []
total_p1_inter  = []
total_p2_inter  = []

delta_p_simple_all = []
delta_p_inter_all  = []

for intens in intensidades_s05:
    factor = intens / (-0.10)  # proporción respecto al caso base (-10%)
    
    dd = np.zeros(40)
    dd[4] = intens  * totales_r[4]
    dd[5] = factor * 0.033 * totales_r[5]
    dd[6] = factor * 0.033 * totales_r[6]
    dd[7] = factor * 0.033 * totales_r[7]
    
    dp_s = deltaProduccion(L_simple, dd)
    dp_i = deltaProduccion(L_interregional, dd)
    dp_s2 = np.linalg.inv(Is - Ass) @ Asr @ dp_i
    
    total_p1_simple.append(dp_s.sum())
    total_p1_inter.append(dp_i.sum())
    total_p2_inter.append(dp_s2.sum())
    delta_p_simple_all.append(dp_s)
    delta_p_inter_all.append(dp_i)

intensidades_pct = intensidades_s05 * 100

In [ ]:
# ── Gráfico 1: Variación total de producción vs intensidad del shock ─────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# P1 - Argentina
axes[0].plot(intensidades_pct, total_p1_simple, 'b-o', label='Modelo simple')
axes[0].plot(intensidades_pct, total_p1_inter,  'r-s', label='Modelo interregional')
axes[0].set_title('Variación total de producción — P1 (Argentina)')
axes[0].set_xlabel('Intensidad del shock en s05 (%)')
axes[0].set_ylabel('Variación total de producción (M USD)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# P2 - Brasil
axes[1].plot(intensidades_pct, total_p2_inter, 'g-^', label='Modelo interregional')
axes[1].set_title('Variación total de producción — P2 (Brasil)')
axes[1].set_xlabel('Intensidad del shock en s05 (%)')
axes[1].set_ylabel('Variación total de producción (M USD)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Impacto del shock en Argentina sobre ARG y BRA', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 2: Top 10 sectores más afectados (escenario -10%) ────────────────
dp_max_simple = delta_p_simple_all[-1]  # último = intensidad -10%
dp_max_inter  = delta_p_inter_all[-1]

df_top10_afectados = pd.DataFrame({
    'Sector':          nombre_sector,
    'Modelo simple':   dp_max_simple,
    'Modelo inter.':   dp_max_inter
})
df_top10_afectados['abs_inter'] = df_top10_afectados['Modelo inter.'].abs()
df_top10_afectados = df_top10_afectados.nlargest(10, 'abs_inter')

df_top10_melt = df_top10_afectados.melt(
    id_vars='Sector',
    value_vars=['Modelo simple', 'Modelo inter.'],
    var_name='Modelo',
    value_name='Variación'
)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_top10_melt, y='Sector', x='Variación', hue='Modelo', orient='h')
plt.title('Top 10 sectores más afectados (escenario -10% en s05)')
plt.xlabel('Variación de producción (M USD)')
plt.ylabel('')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 3: Heatmap de diferencias entre modelos ─────────────────────────
diferencias = np.array(delta_p_inter_all) - np.array(delta_p_simple_all)

df_heatmap = pd.DataFrame(
    diferencias,
    index=[f'{int(i*100)}%' for i in intensidades_s05],
    columns=nombre_sector
)

plt.figure(figsize=(18, 4))
sns.heatmap(
    df_heatmap,
    cmap='RdBu_r',
    center=0,
    annot=False,
    linewidths=0.3,
    cbar_kws={'label': 'Diferencia (M USD)'}
)
plt.title('Diferencia sector a sector: Modelo interregional − Modelo simple')
plt.xlabel('Sector')
plt.ylabel('Intensidad del shock en s05')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

### Discusión final

**¿Qué aporta incorporar las relaciones interregionales?**

El modelo de región simple (ecuación 5) subestima el impacto real del shock en Argentina porque ignora el **efecto de ida y vuelta**: cuando Argentina se contrae, demanda menos de Brasil; Brasil entonces produce menos y a su vez le demanda menos a Argentina, amplificando el efecto original.

La diferencia entre ambos modelos (visible en el heatmap) representa justamente esta subestimación. Los sectores más interconectados entre ARG y BRA son los que muestran mayor discrepancia entre modelos.

Además, el modelo completo permite cuantificar el **spillover sobre Brasil**: aunque el shock ocurre en Argentina, Brasil también experimenta una caída en su producción, efecto que el modelo simple es completamente incapaz de capturar.

En síntesis, para economías con fuertes lazos comerciales como Argentina y Brasil, el análisis interregional es considerablemente más realista y completo.